# Generador corregido: YOLO26n + TensorRT para NVIDIA Jetson Nano

Este notebook crea desde cero `jetson_app_yolo26_2.zip` para una Jetson Nano con
JetPack 4.6.1 / L4T R32.7.1. Descarga `yolo26n.pt`, exporta **un solo ONNX estático,
simplificado y con opset 12**, valida el archivo y genera:

- inferencia TensorRT FP16 en la GPU de la Jetson;
- cámara USB o CSI;
- servidor FastAPI;
- video MJPEG y resultados por WebSocket;
- página HTML responsiva con detecciones en tiempo real;
- Dockerfile y scripts de instalación/despliegue;
- wheelhouse ARM64/Python 3.6 para instalar FastAPI sin depender de PyPI en la Jetson.

## Por qué no se usa `cv2.dnn.readNetFromONNX`

El OpenCV 4.5.0 incluido en el contenedor oficial de JetPack 4 no puede ejecutar
correctamente el grafo moderno de YOLO26. La cabeza end-to-end usa operaciones como
`TopK`, y la cabeza one-to-many contiene atención/multiplicaciones matriciales que
también fallan durante `forward()` en OpenCV 4.5.x. TensorRT es el backend nativo de
NVIDIA y evita ese importador antiguo. OpenCV sigue usándose para cámara, JPEG y dibujo.

Ejecuta todas las celdas en orden en Google Colab. La última celda descarga el ZIP.


In [1]:
# Dependencias de exportación y validación en Google Colab.
# Se fijan versiones para que el resultado sea reproducible.
%pip -q install \
    "ultralytics==8.4.152" \
    "onnx==1.18.0" \
    "onnxruntime==1.22.1" \
    "onnxslim==0.1.82"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.3/167.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.0 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import ast
import hashlib
import json
import os
import shutil
import subprocess
import sys
import textwrap
import zipfile

import numpy as np
import onnx
import onnxruntime as ort
import ultralytics
from ultralytics import YOLO


# ============================================================
# CONFIGURACIÓN DEL PAQUETE
# ============================================================

MODEL_WEIGHTS = "yolo26n.pt"
MODEL_ONNX_NAME = "yolo26n_jetson.onnx"
MODEL_ENGINE_NAME = "yolo26n_jetson.engine"

# 320 es una elección realista para la memoria/rendimiento de Nano.
# Debe ser múltiplo de 32. Puede cambiarse a 416 o 640 y volver a
# ejecutar todo el notebook.
MODEL_IMAGE_SIZE = 320
ONNX_OPSET = 12

PACKAGE_NAME = "jetson_app_yolo26_2"
ZIP_NAME = PACKAGE_NAME + ".zip"

try:
    import google.colab  # noqa: F401
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

BASE_DIR = Path("/content") if RUNNING_IN_COLAB else Path.cwd()
PROJECT_DIR = BASE_DIR / PACKAGE_NAME
MODELS_DIR = PROJECT_DIR / "models"
WHEELHOUSE_DIR = PROJECT_DIR / "wheelhouse"
ZIP_PATH = BASE_DIR / ZIP_NAME

if MODEL_IMAGE_SIZE <= 0 or MODEL_IMAGE_SIZE % 32 != 0:
    raise ValueError("MODEL_IMAGE_SIZE debe ser positivo y múltiplo de 32")

# Guardia para no borrar una ruta amplia por accidente.
if PROJECT_DIR.name != PACKAGE_NAME:
    raise RuntimeError("Ruta de proyecto inesperada")

if PROJECT_DIR.exists():
    shutil.rmtree(str(PROJECT_DIR))

PROJECT_DIR.mkdir(parents=True)
MODELS_DIR.mkdir(parents=True)
WHEELHOUSE_DIR.mkdir(parents=True)


def write_file(relative_path, content):
    path = PROJECT_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        textwrap.dedent(content).lstrip(),
        encoding="utf-8",
    )
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


print("Ultralytics:", ultralytics.__version__)
print("Directorio del proyecto:", PROJECT_DIR)
print("ZIP final:", ZIP_PATH)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.152
Directorio del proyecto: /content/jetson_app_yolo26_2
ZIP final: /content/jetson_app_yolo26_2.zip


In [3]:
# ============================================================
# DESCARGAR YOLO26n, EXPORTAR UN ÚNICO ONNX Y VALIDARLO
# ============================================================

yolo = YOLO(MODEL_WEIGHTS)

names = yolo.names
if isinstance(names, dict):
    class_names = [names[index] for index in sorted(names)]
else:
    class_names = list(names)

if len(class_names) != 80:
    raise RuntimeError(
        "Se esperaban 80 clases COCO y se obtuvieron {}".format(
            len(class_names)
        )
    )

# nms=None selecciona la cabeza one-to-many. No se incrusta NMS ni
# TopK en el ONNX; NMS se hará después de TensorRT con OpenCV.
exported = yolo.export(
    format="onnx",
    imgsz=MODEL_IMAGE_SIZE,
    batch=1,
    opset=ONNX_OPSET,
    simplify=True,
    dynamic=False,
    nms=None,
    device="cpu",
)

exported_path = Path(str(exported))
if not exported_path.is_absolute():
    exported_path = Path.cwd() / exported_path

if not exported_path.exists():
    search_roots = [Path.cwd(), BASE_DIR]
    candidates = []
    for root in search_roots:
        candidates.extend(root.glob("yolo26n*.onnx"))
    candidates = [path for path in candidates if path.is_file()]
    if not candidates:
        raise RuntimeError("No se encontró el ONNX exportado")
    exported_path = max(candidates, key=lambda path: path.stat().st_mtime)

model_path = MODELS_DIR / MODEL_ONNX_NAME
if exported_path.resolve() != model_path.resolve():
    shutil.copy2(str(exported_path), str(model_path))

labels_path = MODELS_DIR / "coco.names"
labels_path.write_text(
    "\n".join(str(name) for name in class_names) + "\n",
    encoding="utf-8",
)

# Validación estructural ONNX.
graph = onnx.load(str(model_path))
onnx.checker.check_model(graph)

default_opsets = [
    item.version for item in graph.opset_import if item.domain in ("", "ai.onnx")
]
if default_opsets != [ONNX_OPSET]:
    raise RuntimeError("Opset ONNX inesperado: {}".format(default_opsets))

input_dims = [
    dimension.dim_value
    for dimension in graph.graph.input[0].type.tensor_type.shape.dim
]
output_dims = [
    dimension.dim_value
    for dimension in graph.graph.output[0].type.tensor_type.shape.dim
]

expected_candidates = sum(
    (MODEL_IMAGE_SIZE // stride) ** 2 for stride in (8, 16, 32)
)
expected_output = [1, 4 + len(class_names), expected_candidates]

if input_dims != [1, 3, MODEL_IMAGE_SIZE, MODEL_IMAGE_SIZE]:
    raise RuntimeError("Entrada ONNX inesperada: {}".format(input_dims))

if output_dims != expected_output:
    raise RuntimeError(
        "Salida ONNX inesperada: {} != {}".format(
            output_dims,
            expected_output,
        )
    )

# Estas operaciones pertenecen a la selección end-to-end y no deben
# estar presentes en este artefacto para JetPack 4/TensorRT 8.2.
node_types = {node.op_type for node in graph.graph.node}
forbidden_ops = {"TopK", "GatherElements", "NonMaxSuppression"}
found_forbidden = sorted(node_types.intersection(forbidden_ops))
if found_forbidden:
    raise RuntimeError(
        "El ONNX contiene operaciones end-to-end no deseadas: {}".format(
            found_forbidden
        )
    )

# Prueba numérica independiente de Ultralytics.
session = ort.InferenceSession(
    str(model_path),
    providers=["CPUExecutionProvider"],
)
input_name = session.get_inputs()[0].name
probe = np.zeros(
    (1, 3, MODEL_IMAGE_SIZE, MODEL_IMAGE_SIZE),
    dtype=np.float32,
)
prediction = session.run(None, {input_name: probe})[0]

if list(prediction.shape) != expected_output:
    raise RuntimeError(
        "ONNX Runtime entregó una forma inesperada: {}".format(
            prediction.shape
        )
    )
if not np.isfinite(prediction).all():
    raise RuntimeError("La inferencia ONNX produjo NaN o infinito")

model_sha256 = sha256_file(model_path)

model_config = {
    "model_name": "YOLO26n COCO",
    "model_file": MODEL_ONNX_NAME,
    "engine_file": MODEL_ENGINE_NAME,
    "engine_info_file": "engine_info.json",
    "labels_file": "coco.names",
    "input_width": MODEL_IMAGE_SIZE,
    "input_height": MODEL_IMAGE_SIZE,
    "class_count": len(class_names),
    "output_format": "xywh_plus_class_scores",
    "output_shape": expected_output,
    "confidence_threshold": 0.25,
    "nms_threshold": 0.45,
    "min_box_area": 25,
    "max_detections": 50,
    "backend": "tensorrt",
    "onnx_opset": ONNX_OPSET,
    "onnx_sha256": model_sha256,
}

(MODELS_DIR / "model_config.json").write_text(
    json.dumps(model_config, indent=4, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

(MODELS_DIR / "MODEL_SOURCE.txt").write_text(
    "\n".join(
        [
            "Modelo original: yolo26n.pt",
            "Fuente: https://github.com/ultralytics/assets/releases/",
            "Ultralytics: {}".format(ultralytics.__version__),
            "Exportación: ONNX, opset {}, simplify=True, dynamic=False, nms=None".format(
                ONNX_OPSET
            ),
            "Entrada: 1x3x{}x{}".format(MODEL_IMAGE_SIZE, MODEL_IMAGE_SIZE),
            "Salida: {}".format(expected_output),
            "SHA256 ONNX: {}".format(model_sha256),
            "El motor TensorRT se construye en la Jetson; no es portable entre equipos.",
            "",
        ]
    ),
    encoding="utf-8",
)

print("Modelo validado:", model_path)
print("Entrada:", input_dims)
print("Salida:", output_dims)
print("SHA256:", model_sha256)


Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 120 layers, 2,408,932 parameters, 0 gradients, 1.3 GFLOPs

PyTorch: starting from 'yolo26n.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 84, 2100) (5.3 MB)

ONNX: starting export with onnx 1.18.0 opset 12...
ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 9.4s, saved as 'yolo26n.onnx' (9.3 MB)

Export complete (10.1s)
Results saved to /content/yolo26n.onnx
Predict:         yolo predict task=detect model=yolo26n.onnx imgsz=320 
Validate:        yolo val task=detect model=yolo26n.onnx imgsz=320 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app
Modelo validado: /content/jetson_app_yolo26_2/models/yolo26n_jetson.onnx
Entrada: [1, 3, 320, 320]
Salida: [1

In [4]:
# ============================================================
# CÁMARA Y POSPROCESAMIENTO
# ============================================================

write_file(
    "src/__init__.py",
    r'''
    ''',
)

write_file(
    "src/camera.py",
    r'''
    import os
    import cv2


    def _int_env(name, default):
        return int(os.getenv(name, str(default)))


    class Camera(object):
        def __init__(self):
            self.capture = None
            self.description = ""
            self._open()

        def _csi_pipeline(self):
            sensor_id = _int_env("CSI_SENSOR_ID", 0)
            capture_width = _int_env("CSI_CAPTURE_WIDTH", 1280)
            capture_height = _int_env("CSI_CAPTURE_HEIGHT", 720)
            display_width = _int_env("CAMERA_WIDTH", 640)
            display_height = _int_env("CAMERA_HEIGHT", 480)
            fps = _int_env("CAMERA_FPS", 15)
            flip = _int_env("CSI_FLIP_METHOD", 0)

            return (
                "nvarguscamerasrc sensor-id={sensor} ! "
                "video/x-raw(memory:NVMM), "
                "width=(int){capture_width}, height=(int){capture_height}, "
                "format=(string)NV12, framerate=(fraction){fps}/1 ! "
                "nvvidconv flip-method={flip} ! "
                "video/x-raw, width=(int){display_width}, "
                "height=(int){display_height}, format=(string)BGRx ! "
                "videoconvert ! video/x-raw, format=(string)BGR ! "
                "appsink drop=true max-buffers=1 sync=false"
            ).format(
                sensor=sensor_id,
                capture_width=capture_width,
                capture_height=capture_height,
                display_width=display_width,
                display_height=display_height,
                fps=fps,
                flip=flip,
            )

        def _open(self):
            custom_pipeline = os.getenv("GSTREAMER_PIPELINE", "").strip()
            source = os.getenv("CAMERA_SOURCE", "usb").strip().lower()

            if custom_pipeline:
                self.description = "GStreamer personalizado"
                self.capture = cv2.VideoCapture(
                    custom_pipeline,
                    cv2.CAP_GSTREAMER,
                )
            elif source == "csi":
                pipeline = self._csi_pipeline()
                self.description = "CSI/Argus"
                self.capture = cv2.VideoCapture(
                    pipeline,
                    cv2.CAP_GSTREAMER,
                )
            elif source == "usb":
                device = os.getenv("CAMERA_DEVICE", "/dev/video0").strip()
                self.description = "USB {}".format(device)
                self.capture = cv2.VideoCapture(
                    device,
                    cv2.CAP_V4L2,
                )
                self.capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)
                self.capture.set(
                    cv2.CAP_PROP_FRAME_WIDTH,
                    _int_env("CAMERA_WIDTH", 640),
                )
                self.capture.set(
                    cv2.CAP_PROP_FRAME_HEIGHT,
                    _int_env("CAMERA_HEIGHT", 480),
                )
                self.capture.set(
                    cv2.CAP_PROP_FPS,
                    _int_env("CAMERA_FPS", 15),
                )
                if os.getenv("CAMERA_MJPG", "1") == "1":
                    self.capture.set(
                        cv2.CAP_PROP_FOURCC,
                        cv2.VideoWriter_fourcc(*"MJPG"),
                    )
            else:
                raise ValueError(
                    "CAMERA_SOURCE debe ser usb o csi; se recibió {!r}".format(
                        source
                    )
                )

            if self.capture is None or not self.capture.isOpened():
                self.release()
                raise RuntimeError(
                    "No se pudo abrir la cámara ({})".format(self.description)
                )

        def read(self):
            ok, frame = self.capture.read()
            if not ok or frame is None or frame.size == 0:
                return None
            return frame

        def release(self):
            if self.capture is not None:
                self.capture.release()
                self.capture = None
    ''',
)

write_file(
    "src/postprocess.py",
    r'''
    import cv2
    import numpy as np


    def _rows_from_output(output, class_count):
        values = np.asarray(output)

        if values.ndim == 3 and values.shape[0] == 1:
            values = values[0]

        if values.ndim != 2:
            raise RuntimeError(
                "Salida TensorRT no soportada: {}".format(values.shape)
            )

        columns = 4 + int(class_count)

        if values.shape[0] == columns:
            values = values.transpose(1, 0)
        elif values.shape[1] != columns:
            raise RuntimeError(
                "Se esperaban {} columnas y se obtuvo {}".format(
                    columns,
                    values.shape,
                )
            )

        return np.ascontiguousarray(values, dtype=np.float32)


    def _undo_letterbox(
        box,
        ratio,
        pad_x,
        pad_y,
        frame_width,
        frame_height,
    ):
        x1, y1, x2, y2 = box
        x1 = (x1 - pad_x) / ratio
        y1 = (y1 - pad_y) / ratio
        x2 = (x2 - pad_x) / ratio
        y2 = (y2 - pad_y) / ratio

        x1 = max(0, min(frame_width - 1, int(round(x1))))
        y1 = max(0, min(frame_height - 1, int(round(y1))))
        x2 = max(0, min(frame_width - 1, int(round(x2))))
        y2 = max(0, min(frame_height - 1, int(round(y2))))
        return x1, y1, x2, y2


    def decode_yolo26(
        output,
        class_count,
        confidence_threshold,
        nms_threshold,
        max_detections,
        min_box_area,
        ratio,
        pad_x,
        pad_y,
        frame_width,
        frame_height,
    ):
        rows = _rows_from_output(output, class_count)
        candidates = []

        for row in rows:
            scores = row[4:4 + class_count]
            class_id = int(np.argmax(scores))
            confidence = float(scores[class_id])

            if confidence < confidence_threshold:
                continue

            center_x = float(row[0])
            center_y = float(row[1])
            width = float(row[2])
            height = float(row[3])

            x1 = center_x - width * 0.5
            y1 = center_y - height * 0.5
            x2 = center_x + width * 0.5
            y2 = center_y + height * 0.5

            x1, y1, x2, y2 = _undo_letterbox(
                (x1, y1, x2, y2),
                ratio,
                pad_x,
                pad_y,
                frame_width,
                frame_height,
            )

            if x2 <= x1 or y2 <= y1:
                continue
            if (x2 - x1) * (y2 - y1) < min_box_area:
                continue

            candidates.append(
                {
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "class_id": class_id,
                    "confidence": confidence,
                }
            )

        grouped = {}
        for item in candidates:
            grouped.setdefault(item["class_id"], []).append(item)

        selected = []
        for items in grouped.values():
            boxes = [
                [
                    item["x1"],
                    item["y1"],
                    item["x2"] - item["x1"],
                    item["y2"] - item["y1"],
                ]
                for item in items
            ]
            scores = [item["confidence"] for item in items]
            keep = cv2.dnn.NMSBoxes(
                boxes,
                scores,
                float(confidence_threshold),
                float(nms_threshold),
            )

            if keep is None or len(keep) == 0:
                continue

            for index in np.asarray(keep).reshape(-1):
                selected.append(items[int(index)])

        selected.sort(
            key=lambda item: item["confidence"],
            reverse=True,
        )
        return selected[: int(max_detections)]
    ''',
)

print("Cámara y posprocesamiento creados")


Cámara y posprocesamiento creados


In [5]:
# ============================================================
# BACKEND TENSORRT Y SERVIDOR
# ============================================================

write_file(
    "src/model.py",
    r'''
    import json
    import os
    import time
    from pathlib import Path

    import cv2
    import numpy as np

    from src.postprocess import decode_yolo26


    class TensorRTRunner(object):
        def __init__(self, engine_path):
            try:
                import cupy as cp
                import tensorrt as trt
            except ImportError as error:
                raise RuntimeError(
                    "Falta TensorRT o CuPy en el contenedor: {}".format(error)
                )

            self.cp = cp
            self.trt = trt
            self.engine_path = Path(engine_path)

            if not self.engine_path.exists():
                raise RuntimeError(
                    "No existe el motor TensorRT: {}".format(
                        self.engine_path
                    )
                )

            self.logger = trt.Logger(trt.Logger.WARNING)
            self.runtime = trt.Runtime(self.logger)
            self.engine = self.runtime.deserialize_cuda_engine(
                self.engine_path.read_bytes()
            )

            if self.engine is None:
                raise RuntimeError("TensorRT no pudo abrir el motor")

            self.context = self.engine.create_execution_context()
            if self.context is None:
                raise RuntimeError("TensorRT no pudo crear el contexto")

            self.stream = cp.cuda.Stream(non_blocking=True)
            self.bindings = [0] * self.engine.num_bindings
            self.inputs = []
            self.outputs = []

            for index in range(self.engine.num_bindings):
                shape = tuple(
                    int(value)
                    for value in self.engine.get_binding_shape(index)
                )
                if not shape or any(value <= 0 for value in shape):
                    raise RuntimeError(
                        "Binding dinámico/no válido {}: {}".format(
                            index,
                            shape,
                        )
                    )

                dtype = np.dtype(
                    trt.nptype(self.engine.get_binding_dtype(index))
                )
                device_array = cp.empty(shape, dtype=dtype)
                self.bindings[index] = int(device_array.data.ptr)

                item = {
                    "index": index,
                    "name": self.engine.get_binding_name(index),
                    "shape": shape,
                    "dtype": dtype,
                    "device": device_array,
                }

                if self.engine.binding_is_input(index):
                    self.inputs.append(item)
                else:
                    self.outputs.append(item)

            if len(self.inputs) != 1 or len(self.outputs) != 1:
                raise RuntimeError(
                    "Se esperaba 1 entrada y 1 salida; se obtuvo {} y {}".format(
                        len(self.inputs),
                        len(self.outputs),
                    )
                )

            self.input_shape = self.inputs[0]["shape"]
            self.output_shape = self.outputs[0]["shape"]

        def infer(self, blob):
            input_item = self.inputs[0]
            host_input = np.asarray(
                blob,
                dtype=input_item["dtype"],
                order="C",
            )

            if tuple(host_input.shape) != input_item["shape"]:
                raise RuntimeError(
                    "Entrada {} != {}".format(
                        host_input.shape,
                        input_item["shape"],
                    )
                )

            input_item["device"].set(
                host_input,
                stream=self.stream,
            )
            ok = self.context.execute_async_v2(
                self.bindings,
                self.stream.ptr,
            )
            if not ok:
                raise RuntimeError("TensorRT execute_async_v2 devolvió False")

            self.stream.synchronize()
            return self.cp.asnumpy(self.outputs[0]["device"])


    class Detector(object):
        def __init__(
            self,
            config_path="/app/models/model_config.json",
        ):
            self.config_path = Path(config_path)
            with self.config_path.open("r", encoding="utf-8") as source:
                config = json.load(source)

            base = self.config_path.parent
            labels_path = base / config["labels_file"]
            engine_path = base / config["engine_file"]

            with labels_path.open("r", encoding="utf-8") as source:
                self.labels = [
                    line.strip()
                    for line in source.readlines()
                    if line.strip()
                ]

            expected_classes = int(config.get("class_count", 80))
            if len(self.labels) != expected_classes:
                raise RuntimeError(
                    "Etiquetas {} != {}".format(
                        len(self.labels),
                        expected_classes,
                    )
                )

            self.model_name = config.get("model_name", "YOLO26n")
            self.engine_file = engine_path.name
            self.input_width = int(config["input_width"])
            self.input_height = int(config["input_height"])
            self.confidence_threshold = float(
                os.getenv(
                    "YOLO_CONF",
                    str(config.get("confidence_threshold", 0.25)),
                )
            )
            self.nms_threshold = float(
                os.getenv(
                    "YOLO_NMS",
                    str(config.get("nms_threshold", 0.45)),
                )
            )
            self.min_box_area = int(config.get("min_box_area", 25))
            self.max_detections = int(
                config.get("max_detections", 50)
            )

            info_path = base / config.get(
                "engine_info_file",
                "engine_info.json",
            )
            if not info_path.exists():
                log_path = base / "engine_build.log"
                log_tail = ""
                if log_path.exists():
                    try:
                        log_tail = log_path.read_text(
                            encoding="utf-8",
                            errors="replace",
                        )[-1800:]
                    except Exception:
                        log_tail = ""
                raise RuntimeError(
                    "No existe engine_info.json; la construcción TensorRT no terminó. "
                    "Último log: {}".format(log_tail or "sin detalle")
                )

            with info_path.open("r", encoding="utf-8") as source:
                engine_info = json.load(source)

            expected_hash = config.get("onnx_sha256")
            if engine_info.get("onnx_sha256") != expected_hash:
                raise RuntimeError(
                    "El motor TensorRT no corresponde al ONNX incluido"
                )

            precision = engine_info.get("precision", "desconocida")
            self.backend = "TensorRT {}".format(precision)
            self.runner = TensorRTRunner(engine_path)

            expected_input = (
                1,
                3,
                self.input_height,
                self.input_width,
            )
            if self.runner.input_shape != expected_input:
                raise RuntimeError(
                    "Motor con entrada {} != {}".format(
                        self.runner.input_shape,
                        expected_input,
                    )
                )

            self.fps_ema = 0.0

        def _letterbox(self, image):
            height, width = image.shape[:2]
            ratio = min(
                float(self.input_width) / float(width),
                float(self.input_height) / float(height),
            )
            new_width = int(round(width * ratio))
            new_height = int(round(height * ratio))

            resized = cv2.resize(
                image,
                (new_width, new_height),
                interpolation=cv2.INTER_LINEAR,
            )
            pad_width = self.input_width - new_width
            pad_height = self.input_height - new_height
            left = int(round(pad_width / 2.0 - 0.1))
            right = int(round(pad_width / 2.0 + 0.1))
            top = int(round(pad_height / 2.0 - 0.1))
            bottom = int(round(pad_height / 2.0 + 0.1))

            output = cv2.copyMakeBorder(
                resized,
                top,
                bottom,
                left,
                right,
                cv2.BORDER_CONSTANT,
                value=(114, 114, 114),
            )
            return output, ratio, left, top

        @staticmethod
        def _color(class_id):
            palette = (
                (45, 212, 191),
                (96, 165, 250),
                (244, 114, 182),
                (251, 191, 36),
                (167, 139, 250),
                (74, 222, 128),
            )
            rgb = palette[class_id % len(palette)]
            return rgb[2], rgb[1], rgb[0]

        def _draw(self, image, item, text):
            x1 = item["x1"]
            y1 = item["y1"]
            x2 = item["x2"]
            y2 = item["y2"]
            color = self._color(item["class_id"])

            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            font = cv2.FONT_HERSHEY_SIMPLEX
            scale = 0.52
            thickness = 1
            (text_width, text_height), baseline = cv2.getTextSize(
                text,
                font,
                scale,
                thickness,
            )
            pad = 4
            label_top = max(0, y1 - text_height - baseline - pad * 2)
            label_bottom = label_top + text_height + baseline + pad * 2
            label_right = min(
                image.shape[1] - 1,
                x1 + text_width + pad * 2,
            )
            cv2.rectangle(
                image,
                (x1, label_top),
                (label_right, label_bottom),
                color,
                -1,
            )
            cv2.putText(
                image,
                text,
                (x1 + pad, label_top + text_height + pad),
                font,
                scale,
                (10, 15, 25),
                thickness,
                cv2.LINE_AA,
            )

        def analyze(self, frame):
            total_start = time.perf_counter()
            frame_height, frame_width = frame.shape[:2]
            prepared, ratio, pad_x, pad_y = self._letterbox(frame)
            blob = cv2.dnn.blobFromImage(
                prepared,
                scalefactor=1.0 / 255.0,
                size=(self.input_width, self.input_height),
                mean=(0, 0, 0),
                swapRB=True,
                crop=False,
            )

            inference_start = time.perf_counter()
            output = self.runner.infer(blob)
            inference_ms = (
                time.perf_counter() - inference_start
            ) * 1000.0

            detections = decode_yolo26(
                output=output,
                class_count=len(self.labels),
                confidence_threshold=self.confidence_threshold,
                nms_threshold=self.nms_threshold,
                max_detections=self.max_detections,
                min_box_area=self.min_box_area,
                ratio=ratio,
                pad_x=pad_x,
                pad_y=pad_y,
                frame_width=frame_width,
                frame_height=frame_height,
            )

            result_frame = frame.copy()
            public_boxes = []
            for item in detections:
                class_name = self.labels[item["class_id"]]
                label = "{} {:.0f}%".format(
                    class_name,
                    item["confidence"] * 100.0,
                )
                self._draw(result_frame, item, label)
                public_boxes.append(
                    {
                        "x": item["x1"],
                        "y": item["y1"],
                        "width": item["x2"] - item["x1"],
                        "height": item["y2"] - item["y1"],
                        "x1": item["x1"],
                        "y1": item["y1"],
                        "x2": item["x2"],
                        "y2": item["y2"],
                        "class_id": item["class_id"],
                        "name": class_name,
                        "class_name": class_name,
                        "confidence": round(item["confidence"], 4),
                        "label": label,
                    }
                )

            total_ms = (
                time.perf_counter() - total_start
            ) * 1000.0
            instant_fps = 1000.0 / total_ms if total_ms > 0 else 0.0
            if self.fps_ema <= 0:
                self.fps_ema = instant_fps
            else:
                self.fps_ema = 0.85 * self.fps_ema + 0.15 * instant_fps

            return result_frame, {
                "status": "ok",
                "stage": "running",
                "timestamp": time.time(),
                "model": self.model_name,
                "model_file": self.engine_file,
                "backend": self.backend,
                "inference_ms": round(inference_ms, 2),
                "total_ms": round(total_ms, 2),
                "fps": round(self.fps_ema, 2),
                "objects": len(public_boxes),
                "boxes": public_boxes,
                "detections": public_boxes,
            }
    ''',
)

write_file(
    "src/server.py",
    r'''
    import asyncio
    import os
    import threading
    import time

    import cv2
    from fastapi import FastAPI, WebSocket
    from fastapi.responses import StreamingResponse
    from fastapi.staticfiles import StaticFiles
    from starlette.websockets import WebSocketDisconnect

    from src.camera import Camera
    from src.model import Detector


    app = FastAPI(title="YOLO26 Jetson Nano")
    stop_event = threading.Event()
    raw_lock = threading.Lock()
    result_lock = threading.Lock()

    raw_frame = None
    raw_sequence = 0
    latest_frame = b""
    latest_result = {
        "status": "starting",
        "stage": "server",
        "model": "YOLO26n COCO",
        "objects": 0,
        "boxes": [],
        "detections": [],
    }


    def set_state(**changes):
        global latest_result
        with result_lock:
            updated = dict(latest_result)
            updated.update(changes)
            latest_result = updated


    def set_error(stage, error):
        set_state(
            status="error",
            stage=stage,
            error=str(error),
            objects=0,
            boxes=[],
            detections=[],
        )


    def capture_loop():
        global raw_frame
        global raw_sequence

        retry_seconds = float(os.getenv("CAMERA_RETRY_SECONDS", "2"))

        while not stop_event.is_set():
            camera = None
            try:
                set_state(
                    status="starting",
                    stage="camera",
                    error=None,
                )
                camera = Camera()
                set_state(
                    status="starting",
                    stage="waiting_frame",
                    camera=camera.description,
                    error=None,
                )
                failed_reads = 0

                while not stop_event.is_set():
                    frame = camera.read()
                    if frame is None:
                        failed_reads += 1
                        if failed_reads >= 30:
                            raise RuntimeError(
                                "La cámara dejó de entregar imágenes"
                            )
                        time.sleep(0.02)
                        continue

                    failed_reads = 0
                    with raw_lock:
                        raw_frame = frame
                        raw_sequence += 1

            except Exception as error:
                set_error("camera", error)
                stop_event.wait(retry_seconds)
            finally:
                if camera is not None:
                    camera.release()


    def inference_loop():
        global latest_frame
        global latest_result

        try:
            set_state(
                status="starting",
                stage="loading_model",
                error=None,
            )
            detector = Detector()
            set_state(
                status="starting",
                stage="waiting_camera",
                backend=detector.backend,
                model_file=detector.engine_file,
                error=None,
            )
        except Exception as error:
            set_error("model", error)
            return

        last_sequence = -1
        jpeg_quality = int(os.getenv("JPEG_QUALITY", "75"))

        while not stop_event.is_set():
            with raw_lock:
                if raw_frame is None or raw_sequence == last_sequence:
                    frame = None
                    sequence = last_sequence
                else:
                    frame = raw_frame.copy()
                    sequence = raw_sequence

            if frame is None:
                stop_event.wait(0.01)
                continue

            last_sequence = sequence

            try:
                processed, result = detector.analyze(frame)
                ok, encoded = cv2.imencode(
                    ".jpg",
                    processed,
                    [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality],
                )
                if not ok:
                    raise RuntimeError("OpenCV no pudo codificar JPEG")

                with result_lock:
                    latest_frame = encoded.tobytes()
                    latest_result = result
            except Exception as error:
                set_error("inference", error)
                stop_event.wait(0.25)


    @app.on_event("startup")
    def startup():
        stop_event.clear()
        threading.Thread(
            target=capture_loop,
            name="camera",
            daemon=True,
        ).start()
        threading.Thread(
            target=inference_loop,
            name="inference",
            daemon=True,
        ).start()


    @app.on_event("shutdown")
    def shutdown():
        stop_event.set()


    @app.get("/api/health")
    def health():
        with result_lock:
            result = dict(latest_result)
        result["service_alive"] = True
        return result


    @app.get("/api/latest")
    def latest():
        with result_lock:
            return dict(latest_result)


    @app.get("/stream.mjpg")
    def stream():
        def generate():
            while not stop_event.is_set():
                with result_lock:
                    frame = latest_frame
                if frame:
                    yield (
                        b"--frame\r\n"
                        b"Content-Type: image/jpeg\r\n"
                        b"Cache-Control: no-store\r\n\r\n"
                        + frame
                        + b"\r\n"
                    )
                stop_event.wait(0.03)

        return StreamingResponse(
            generate(),
            media_type="multipart/x-mixed-replace; boundary=frame",
            headers={
                "Cache-Control": "no-store, no-cache, must-revalidate",
                "Pragma": "no-cache",
            },
        )


    @app.websocket("/ws")
    async def websocket_endpoint(websocket: WebSocket):
        await websocket.accept()
        try:
            while True:
                with result_lock:
                    result = dict(latest_result)
                await websocket.send_json(result)
                await asyncio.sleep(0.20)
        except WebSocketDisconnect:
            pass
        except Exception:
            pass


    app.mount(
        "/",
        StaticFiles(directory="/app/static", html=True),
        name="static",
    )
    ''',
)

print("Backend TensorRT y servidor creados")


Backend TensorRT y servidor creados


In [6]:
# ============================================================
# INTERFAZ HTML
# ============================================================

write_file(
    "static/index.html",
    r'''
    <!doctype html>
    <html lang="es">
    <head>
        <meta charset="utf-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <meta name="color-scheme" content="dark">
        <title>YOLO26 · Jetson Nano</title>
        <style>
            :root {
                --bg: #07111f;
                --panel: rgba(15, 30, 50, 0.88);
                --line: rgba(148, 163, 184, 0.18);
                --text: #e5eefb;
                --muted: #9fb0c8;
                --blue: #60a5fa;
                --green: #2dd4bf;
                --red: #fb7185;
            }
            * { box-sizing: border-box; }
            body {
                margin: 0;
                min-height: 100vh;
                color: var(--text);
                font-family: Inter, ui-sans-serif, system-ui, -apple-system,
                    BlinkMacSystemFont, "Segoe UI", sans-serif;
                background:
                    radial-gradient(circle at 15% 0%, #123260 0, transparent 36%),
                    radial-gradient(circle at 90% 12%, #123b34 0, transparent 30%),
                    var(--bg);
            }
            main {
                width: min(1180px, calc(100% - 28px));
                margin: 0 auto;
                padding: 26px 0 42px;
            }
            header {
                display: flex;
                align-items: flex-end;
                justify-content: space-between;
                gap: 18px;
                margin-bottom: 18px;
            }
            h1, h2, p { margin-top: 0; }
            h1 { margin-bottom: 6px; font-size: clamp(1.55rem, 4vw, 2.5rem); }
            h1 span { color: var(--blue); }
            .subtitle { color: var(--muted); margin-bottom: 0; }
            .status {
                display: inline-flex;
                align-items: center;
                gap: 9px;
                min-width: 160px;
                padding: 10px 13px;
                border: 1px solid var(--line);
                border-radius: 999px;
                background: var(--panel);
                font-size: 0.9rem;
            }
            .dot {
                width: 10px;
                height: 10px;
                border-radius: 50%;
                background: #fbbf24;
                box-shadow: 0 0 14px currentColor;
            }
            .status.ok .dot { background: var(--green); }
            .status.error .dot { background: var(--red); }
            .grid {
                display: grid;
                grid-template-columns: minmax(0, 2fr) minmax(250px, 0.8fr);
                gap: 18px;
            }
            .panel {
                border: 1px solid var(--line);
                border-radius: 18px;
                background: var(--panel);
                box-shadow: 0 18px 50px rgba(0, 0, 0, 0.24);
                overflow: hidden;
            }
            .video-wrap {
                position: relative;
                min-height: 320px;
                display: grid;
                place-items: center;
                background: #020617;
            }
            #video {
                display: block;
                width: 100%;
                height: auto;
                min-height: 320px;
                object-fit: contain;
            }
            .metrics {
                display: grid;
                grid-template-columns: repeat(4, 1fr);
                gap: 1px;
                background: var(--line);
            }
            .metric { padding: 14px; background: #0b1728; }
            .metric strong { display: block; font-size: 1.2rem; }
            .metric span { color: var(--muted); font-size: 0.76rem; }
            .side { padding: 18px; }
            .side h2 { font-size: 1rem; color: var(--muted); }
            #detections { display: grid; gap: 9px; }
            .detection, .empty, .error-box {
                padding: 12px 13px;
                border-radius: 11px;
                border: 1px solid var(--line);
                background: rgba(30, 58, 86, 0.56);
            }
            .detection {
                display: flex;
                justify-content: space-between;
                gap: 12px;
            }
            .detection b { color: var(--green); }
            .empty { color: var(--muted); }
            .error-box { color: #fecdd3; border-color: rgba(251, 113, 133, 0.35); }
            .details { margin-top: 14px; color: var(--muted); font-size: 0.8rem; }
            pre { white-space: pre-wrap; word-break: break-word; font-size: 0.72rem; }
            @media (max-width: 800px) {
                header { align-items: flex-start; flex-direction: column; }
                .grid { grid-template-columns: 1fr; }
                .metrics { grid-template-columns: repeat(2, 1fr); }
            }
        </style>
    </head>
    <body>
        <main>
            <header>
                <div>
                    <h1><span>YOLO26</span> · Jetson Nano</h1>
                    <p class="subtitle">Detección COCO con TensorRT en tiempo real</p>
                </div>
                <div id="status" class="status">
                    <span class="dot"></span>
                    <span id="statusText">Conectando…</span>
                </div>
            </header>

            <div class="grid">
                <section class="panel">
                    <div class="video-wrap">
                        <img id="video" src="/stream.mjpg" alt="Cámara con detecciones">
                    </div>
                    <div class="metrics">
                        <div class="metric"><strong id="fps">—</strong><span>FPS proceso</span></div>
                        <div class="metric"><strong id="inference">—</strong><span>Inferencia</span></div>
                        <div class="metric"><strong id="objects">0</strong><span>Objetos</span></div>
                        <div class="metric"><strong id="backend">—</strong><span>Backend</span></div>
                    </div>
                </section>

                <aside class="panel side">
                    <h2>Objetos detectados</h2>
                    <div id="detections"><div class="empty">Esperando imágenes…</div></div>
                    <details class="details">
                        <summary>Diagnóstico JSON</summary>
                        <pre id="json"></pre>
                    </details>
                </aside>
            </div>
        </main>

        <script>
            const status = document.getElementById("status");
            const statusText = document.getElementById("statusText");
            const detections = document.getElementById("detections");
            const jsonElement = document.getElementById("json");
            const fps = document.getElementById("fps");
            const inference = document.getElementById("inference");
            const objects = document.getElementById("objects");
            const backend = document.getElementById("backend");
            let reconnectDelay = 800;

            function setStatus(kind, message) {
                status.className = "status " + kind;
                statusText.textContent = message;
            }

            function render(data) {
                const boxes = Array.isArray(data.boxes) ? data.boxes : [];
                detections.replaceChildren();

                if (data.status === "error") {
                    const error = document.createElement("div");
                    error.className = "error-box";
                    error.textContent = (data.stage || "error") + ": " + (data.error || "sin detalle");
                    detections.appendChild(error);
                    setStatus("error", "Revisar diagnóstico");
                } else if (boxes.length === 0) {
                    const empty = document.createElement("div");
                    empty.className = "empty";
                    empty.textContent = data.status === "ok"
                        ? "No hay objetos sobre el umbral."
                        : "Inicializando " + (data.stage || "servicio") + "…";
                    detections.appendChild(empty);
                    setStatus(data.status === "ok" ? "ok" : "", data.status === "ok" ? "Activo" : "Inicializando…");
                } else {
                    for (const box of boxes) {
                        const row = document.createElement("div");
                        row.className = "detection";
                        const name = document.createElement("b");
                        const confidence = document.createElement("span");
                        name.textContent = box.name || "objeto";
                        confidence.textContent = (Number(box.confidence || 0) * 100).toFixed(0) + "%";
                        row.append(name, confidence);
                        detections.appendChild(row);
                    }
                    setStatus("ok", "Detectando");
                }

                fps.textContent = Number(data.fps || 0).toFixed(1);
                inference.textContent = Number(data.inference_ms || 0).toFixed(0) + " ms";
                objects.textContent = String(data.objects || 0);
                backend.textContent = data.backend || "—";
                jsonElement.textContent = JSON.stringify(data, null, 2);
            }

            function connect() {
                const protocol = location.protocol === "https:" ? "wss" : "ws";
                const socket = new WebSocket(protocol + "://" + location.host + "/ws");

                socket.onopen = () => { reconnectDelay = 800; };
                socket.onmessage = (event) => {
                    try { render(JSON.parse(event.data)); }
                    catch (error) { setStatus("error", "JSON inválido"); }
                };
                socket.onclose = () => {
                    setStatus("", "Reconectando…");
                    window.setTimeout(connect, reconnectDelay);
                    reconnectDelay = Math.min(reconnectDelay * 1.5, 5000);
                };
                socket.onerror = () => socket.close();
            }

            document.getElementById("video").addEventListener("error", () => {
                setStatus("error", "Sin flujo de video");
            });
            connect();
        </script>
    </body>
    </html>
    ''',
)

print("Interfaz HTML creada")


Interfaz HTML creada


In [7]:
# ============================================================
# CONSTRUCCIÓN DEL MOTOR TENSORRT EN LA JETSON
# ============================================================

write_file(
    "scripts/prepare_engine.py",
    r'''
    import hashlib
    import json
    import os
    import sys
    import time
    from pathlib import Path

    import tensorrt as trt


    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open("rb") as source:
            for chunk in iter(lambda: source.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()


    def load_json(path):
        with path.open("r", encoding="utf-8") as source:
            return json.load(source)


    def existing_engine_is_valid(engine_path, info_path, expected_hash, logger):
        if not engine_path.exists() or not info_path.exists():
            return False
        try:
            info = load_json(info_path)
            if info.get("onnx_sha256") != expected_hash:
                return False
            if info.get("tensorrt_version") != trt.__version__:
                return False
            runtime = trt.Runtime(logger)
            engine = runtime.deserialize_cuda_engine(engine_path.read_bytes())
            return engine is not None
        except Exception as error:
            print("Motor previo no reutilizable:", error)
            return False


    def main():
        config_path = Path(
            os.getenv(
                "MODEL_CONFIG",
                "/app/models/model_config.json",
            )
        )
        config = load_json(config_path)
        models_dir = config_path.parent
        onnx_path = models_dir / config["model_file"]
        engine_path = models_dir / config["engine_file"]
        info_path = models_dir / config.get(
            "engine_info_file",
            "engine_info.json",
        )

        if not onnx_path.exists():
            raise RuntimeError("No existe {}".format(onnx_path))

        onnx_hash = sha256_file(onnx_path)
        configured_hash = config.get("onnx_sha256")
        if configured_hash and configured_hash != onnx_hash:
            raise RuntimeError("El SHA256 del ONNX no coincide con la configuración")

        logger = trt.Logger(trt.Logger.INFO)
        if existing_engine_is_valid(
            engine_path,
            info_path,
            onnx_hash,
            logger,
        ):
            print("Motor TensorRT válido; se reutiliza:", engine_path)
            return 0

        workspace_mb = int(os.getenv("TRT_WORKSPACE_MB", "512"))
        want_fp16 = os.getenv("TRT_FP16", "1") == "1"
        explicit_batch = 1 << int(
            trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH
        )

        print("Construyendo motor TensorRT. La primera vez puede tardar varios minutos.")
        print("ONNX:", onnx_path)
        print("TensorRT:", trt.__version__)
        print("Workspace MiB:", workspace_mb)

        started = time.time()
        builder = trt.Builder(logger)
        network = builder.create_network(explicit_batch)
        parser = trt.OnnxParser(network, logger)

        if not parser.parse(onnx_path.read_bytes()):
            errors = []
            for index in range(parser.num_errors):
                errors.append(str(parser.get_error(index)))
            raise RuntimeError(
                "TensorRT no pudo parsear el ONNX:\n" + "\n".join(errors)
            )

        build_config = builder.create_builder_config()
        build_config.max_workspace_size = workspace_mb * 1024 * 1024

        use_fp16 = bool(want_fp16 and builder.platform_has_fast_fp16)
        if use_fp16:
            build_config.set_flag(trt.BuilderFlag.FP16)

        engine = builder.build_engine(network, build_config)
        if engine is None:
            raise RuntimeError(
                "TensorRT no pudo construir el motor. "
                "Pruebe TRT_WORKSPACE_MB=768 o revise los logs."
            )

        serialized = engine.serialize()
        temp_engine = Path(str(engine_path) + ".tmp")
        with temp_engine.open("wb") as destination:
            destination.write(serialized)
        os.replace(str(temp_engine), str(engine_path))

        bindings = []
        for index in range(engine.num_bindings):
            bindings.append(
                {
                    "name": engine.get_binding_name(index),
                    "input": bool(engine.binding_is_input(index)),
                    "shape": [
                        int(value)
                        for value in engine.get_binding_shape(index)
                    ],
                    "dtype": str(engine.get_binding_dtype(index)),
                }
            )

        info = {
            "model": config.get("model_name", "YOLO26n"),
            "onnx_sha256": onnx_hash,
            "tensorrt_version": trt.__version__,
            "precision": "FP16" if use_fp16 else "FP32",
            "workspace_mb": workspace_mb,
            "build_seconds": round(time.time() - started, 2),
            "bindings": bindings,
        }
        temp_info = Path(str(info_path) + ".tmp")
        temp_info.write_text(
            json.dumps(info, indent=4) + "\n",
            encoding="utf-8",
        )
        os.replace(str(temp_info), str(info_path))

        print("Motor creado:", engine_path)
        print(json.dumps(info, indent=2))
        return 0


    if __name__ == "__main__":
        try:
            sys.exit(main())
        except Exception as error:
            print("ERROR_PREPARANDO_TENSORRT:", error, file=sys.stderr)
            sys.exit(1)
    ''',
)

write_file(
    "scripts/entrypoint.sh",
    r'''
    #!/usr/bin/env bash
    set -euo pipefail

    echo "Verificando runtime NVIDIA..."
    python3 - <<'PY'
    import cv2
    import cupy as cp
    import tensorrt as trt

    devices = cp.cuda.runtime.getDeviceCount()
    if devices < 1:
        raise RuntimeError(
            "CUDA no ve ninguna GPU. Inicie Docker con --runtime nvidia."
        )

    print("OpenCV:", cv2.__version__)
    print("CuPy:", cp.__version__)
    print("TensorRT:", trt.__version__)
    print("GPU CUDA detectadas:", devices)
    PY

    ENGINE_LOG=/app/models/engine_build.log
    if python3 /app/scripts/prepare_engine.py 2>&1 | tee "$ENGINE_LOG"; then
        echo "Motor TensorRT listo."
    else
        echo "ADVERTENCIA: falló la preparación del motor."
        echo "El servidor seguirá activo y mostrará el error en /api/health."
    fi

    exec python3 -m uvicorn \
        src.server:app \
        --host 0.0.0.0 \
        --port "${PORT:-8000}" \
        --workers 1 \
        --log-level info
    ''',
)

write_file(
    "requirements-jetson.txt",
    r'''
    fastapi==0.63.0
    starlette==0.13.6
    pydantic==1.8.2
    uvicorn==0.13.4
    websockets==9.1
    aiofiles==0.8.0
    click==7.1.2
    h11==0.13.0
    typing-extensions==4.1.1
    ''',
)

print("Constructor TensorRT y entrypoint creados")


Constructor TensorRT y entrypoint creados


In [8]:
# ============================================================
# DOCKER Y DESPLIEGUE
# ============================================================

write_file(
    "Dockerfile",
    r'''
    ARG L4T_TAG=r32.7.1
    FROM nvcr.io/nvidia/l4t-ml:${L4T_TAG}-py3

    ENV PYTHONUNBUFFERED=1 \
        PYTHONIOENCODING=utf-8 \
        LANG=C.UTF-8 \
        LC_ALL=C.UTF-8 \
        OPENBLAS_NUM_THREADS=1

    WORKDIR /app

    COPY requirements-jetson.txt /tmp/requirements-jetson.txt
    COPY wheelhouse /opt/wheelhouse

    RUN python3 -m pip install \
            --no-cache-dir \
            --no-index \
            --find-links=/opt/wheelhouse \
            -r /tmp/requirements-jetson.txt \
        && python3 -c "import fastapi, uvicorn, websockets" \
        && rm -rf /root/.cache/pip

    COPY src /app/src
    COPY static /app/static
    COPY scripts /app/scripts
    COPY models /app/models

    RUN chmod +x /app/scripts/entrypoint.sh

    EXPOSE 8000
    CMD ["bash", "/app/scripts/entrypoint.sh"]
    ''',
)

write_file(
    "start_on_jetson.sh",
    r'''
    #!/usr/bin/env bash
    set -euo pipefail

    SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
    REQUESTED_IMAGE="${JETSON_IMAGE:-jetson-app:1.0}"
    BUILD_IMAGE="${JETSON_BUILD_IMAGE:-jetson-yolo26:2.0}"
    CONTAINER="${JETSON_CONTAINER:-jetson-yolo26}"
    L4T_TAG="${L4T_TAG:-r32.7.1}"

    if ! command -v docker >/dev/null 2>&1; then
        echo "ERROR: Docker no está instalado."
        exit 2
    fi

    image_works() {
        sudo docker run --rm \
            --runtime nvidia \
            "$1" \
            python3 -c \
            "import cv2, cupy, tensorrt, fastapi, uvicorn" \
            >/dev/null 2>&1
    }

    IMAGE=""
    if sudo docker image inspect "$REQUESTED_IMAGE" >/dev/null 2>&1; then
        if image_works "$REQUESTED_IMAGE"; then
            IMAGE="$REQUESTED_IMAGE"
            echo "Reutilizando imagen compatible: $IMAGE"
        else
            echo "La imagen $REQUESTED_IMAGE no tiene toda la pila requerida."
        fi
    fi

    if [ -z "$IMAGE" ]; then
        echo "Construyendo imagen $BUILD_IMAGE..."
        echo "El wheelhouse ya viene en el ZIP; solo debe existir la imagen base L4T."
        sudo docker build \
            --build-arg "L4T_TAG=$L4T_TAG" \
            -t "$BUILD_IMAGE" \
            "$SCRIPT_DIR"
        IMAGE="$BUILD_IMAGE"

        if ! image_works "$IMAGE"; then
            echo "ERROR: la imagen construida no ve TensorRT/CuPy/CUDA."
            exit 3
        fi
    fi

    CAMERA_SOURCE="${CAMERA_SOURCE:-auto}"
    CAMERA_DEVICE="${CAMERA_DEVICE:-}"

    if [ "$CAMERA_SOURCE" = "auto" ]; then
        CAMERA_SOURCE="csi"
        for device in /dev/video*; do
            if [ -e "$device" ]; then
                CAMERA_SOURCE="usb"
                CAMERA_DEVICE="$device"
                break
            fi
        done
    fi

    RUN_ARGS=(
        --name "$CONTAINER"
        --restart unless-stopped
        --runtime nvidia
        --network host
        --ipc host
        -e LANG=C.UTF-8
        -e LC_ALL=C.UTF-8
        -e PYTHONIOENCODING=utf-8
        -e CAMERA_SOURCE="$CAMERA_SOURCE"
        -e CAMERA_WIDTH="${CAMERA_WIDTH:-640}"
        -e CAMERA_HEIGHT="${CAMERA_HEIGHT:-480}"
        -e CAMERA_FPS="${CAMERA_FPS:-15}"
        -e YOLO_CONF="${YOLO_CONF:-0.25}"
        -e YOLO_NMS="${YOLO_NMS:-0.45}"
        -e TRT_FP16="${TRT_FP16:-1}"
        -e TRT_WORKSPACE_MB="${TRT_WORKSPACE_MB:-512}"
        -e JPEG_QUALITY="${JPEG_QUALITY:-75}"
        -v "$SCRIPT_DIR/src:/app/src:ro"
        -v "$SCRIPT_DIR/static:/app/static:ro"
        -v "$SCRIPT_DIR/scripts:/app/scripts:ro"
        -v "$SCRIPT_DIR/models:/app/models"
    )

    if [ "$CAMERA_SOURCE" = "usb" ]; then
        if [ -z "$CAMERA_DEVICE" ] || [ ! -e "$CAMERA_DEVICE" ]; then
            echo "ERROR: cámara USB no encontrada: ${CAMERA_DEVICE:-vacío}"
            exit 4
        fi
        RUN_ARGS+=(
            --device "$CAMERA_DEVICE:/dev/video0"
            -e CAMERA_DEVICE=/dev/video0
        )
    elif [ "$CAMERA_SOURCE" = "csi" ]; then
        if [ ! -S /tmp/argus_socket ]; then
            echo "ERROR: /tmp/argus_socket no existe; Argus no está disponible."
            exit 5
        fi
        RUN_ARGS+=(
            -v /tmp/argus_socket:/tmp/argus_socket
            -e CSI_SENSOR_ID="${CSI_SENSOR_ID:-0}"
            -e CSI_FLIP_METHOD="${CSI_FLIP_METHOD:-0}"
        )
    else
        echo "ERROR: CAMERA_SOURCE debe ser auto, usb o csi."
        exit 6
    fi

    sudo docker rm -f "$CONTAINER" >/dev/null 2>&1 || true

    sudo docker run -d \
        "${RUN_ARGS[@]}" \
        "$IMAGE" \
        bash /app/scripts/entrypoint.sh

    echo
    echo "Contenedor iniciado: $CONTAINER"
    echo "Cámara: $CAMERA_SOURCE ${CAMERA_DEVICE:-}"
    echo "La primera construcción del motor TensorRT puede tardar varios minutos."
    echo "Logs: sudo docker logs -f $CONTAINER"
    echo "Página: http://IP_DE_LA_JETSON:8000"
    ''',
)

write_file(
    "diagnose_on_jetson.sh",
    r'''
    #!/usr/bin/env bash
    set -u

    SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
    CONTAINER="${JETSON_CONTAINER:-jetson-yolo26}"

    echo "=== Jetson ==="
    uname -a
    if [ -f /etc/nv_tegra_release ]; then
        cat /etc/nv_tegra_release
    fi

    echo "=== Cámaras ==="
    for device in /dev/video*; do
        [ -e "$device" ] && ls -l "$device"
    done

    echo "=== Modelos ==="
    ls -lh "$SCRIPT_DIR/models"

    echo "=== Contenedor ==="
    sudo docker ps -a --filter "name=^/${CONTAINER}$"

    echo "=== Últimos logs ==="
    sudo docker logs --tail 160 "$CONTAINER" 2>&1 || true

    echo "=== API ==="
    if command -v curl >/dev/null 2>&1; then
        curl -fsS http://127.0.0.1:8000/api/health || true
        echo
    fi
    ''',
)

write_file(
    "stop_on_jetson.sh",
    r'''
    #!/usr/bin/env bash
    set -euo pipefail
    sudo docker rm -f "${JETSON_CONTAINER:-jetson-yolo26}"
    ''',
)

write_file(
    "deploy_to_jetson.ps1",
    r'''
    param(
        [Parameter(Mandatory=$true)]
        [string]$PackagePath,

        [string]$JetsonIp = "192.168.55.1",
        [string]$JetsonUser = "artemio",
        [string]$DockerImage = "jetson-app:1.0",
        [string]$BindAddress = "",
        [int]$SshPort = 22
    )

    $ErrorActionPreference = "Stop"

    if ($JetsonUser -notmatch '^[a-z_][a-z0-9_-]*$') {
        throw "JetsonUser contiene caracteres no permitidos"
    }
    if ($DockerImage -notmatch '^[A-Za-z0-9._/:@-]+$') {
        throw "DockerImage contiene caracteres no permitidos"
    }
    if ($SshPort -lt 1 -or $SshPort -gt 65535) {
        throw "SshPort no válido"
    }

    $package = (Resolve-Path -LiteralPath $PackagePath).Path
    $remote = "${JetsonUser}@${JetsonIp}"
    $remoteZip = "/home/$JetsonUser/jetson_app_yolo26_2.zip"
    $remoteDir = "/home/$JetsonUser/jetson_app_yolo26_2"
    $newDir = "${remoteDir}.new"

    $common = @(
        "-4",
        "-o", "ConnectTimeout=30",
        "-o", "ConnectionAttempts=3",
        "-o", "ServerAliveInterval=5"
    )
    if (-not [string]::IsNullOrWhiteSpace($BindAddress)) {
        $common += @("-o", "BindAddress=$BindAddress")
    }

    Write-Host "Transfiriendo paquete..."
    & scp.exe -O @common -P $SshPort $package "${remote}:$remoteZip"
    if ($LASTEXITCODE -ne 0) {
        throw "Falló la transferencia SCP"
    }

    $remoteCommand =
        "set -e; " +
        "rm -rf '$newDir'; " +
        "mkdir -p '$newDir'; " +
        "python3 -m zipfile -e '$remoteZip' '$newDir'; " +
        "test -f '$newDir/models/yolo26n_jetson.onnx'; " +
        "test -f '$newDir/start_on_jetson.sh'; " +
        "if [ -f '$remoteDir/models/yolo26n_jetson.engine' ]; then " +
        "cp '$remoteDir/models/yolo26n_jetson.engine' '$newDir/models/' || true; fi; " +
        "if [ -f '$remoteDir/models/engine_info.json' ]; then " +
        "cp '$remoteDir/models/engine_info.json' '$newDir/models/' || true; fi; " +
        "sudo docker rm -f jetson-yolo26 >/dev/null 2>&1 || true; " +
        "rm -rf '$remoteDir'; " +
        "mv '$newDir' '$remoteDir'; " +
        "chmod +x '$remoteDir/'*.sh '$remoteDir/scripts/'*.sh; " +
        "cd '$remoteDir'; " +
        "JETSON_IMAGE='$DockerImage' bash ./start_on_jetson.sh"

    Write-Host "Instalando e iniciando en la Jetson..."
    & ssh.exe -tt @common -p $SshPort $remote $remoteCommand
    if ($LASTEXITCODE -ne 0) {
        throw "Falló el despliegue remoto"
    }

    Write-Host "Página: http://${JetsonIp}:8000"
    Write-Host "La primera vez siga los logs hasta que aparezca 'Uvicorn running'."
    ''',
)

for executable in (
    PROJECT_DIR / "start_on_jetson.sh",
    PROJECT_DIR / "diagnose_on_jetson.sh",
    PROJECT_DIR / "stop_on_jetson.sh",
    PROJECT_DIR / "scripts/entrypoint.sh",
):
    executable.chmod(0o755)

print("Docker y scripts de despliegue creados")


Docker y scripts de despliegue creados


In [9]:
# ============================================================
# DOCUMENTACIÓN Y DEPENDENCIAS OFFLINE PARA ARM64/PYTHON 3.6
# ============================================================

write_file(
    "README.md",
    r'''
    # YOLO26n + TensorRT para NVIDIA Jetson Nano

    Este paquete fue generado para Jetson Nano con JetPack 4.6.1
    (L4T R32.7.1). Incluye el modelo ONNX y las dependencias web ARM64.

    ## Cambio técnico importante

    OpenCV 4.5.0 no se usa para cargar YOLO26. Ese importador ONNX es
    demasiado antiguo para el grafo de YOLO26. TensorRT ejecuta la red
    en la GPU; OpenCV solo captura la cámara, dibuja y codifica JPEG.

    No existe `yolo26n_raw.onnx` ni `yolo26n_end2end.onnx` en este ZIP.
    El único modelo es `models/yolo26n_jetson.onnx`.

    ## Requisitos

    - NVIDIA Jetson Nano con JetPack 4.6.1 / L4T R32.7.1.
    - Docker y NVIDIA Container Runtime funcionando.
    - Cámara USB UVC o cámara CSI compatible con Argus.
    - Para construir una imagen nueva, debe existir localmente
      `nvcr.io/nvidia/l4t-ml:r32.7.1-py3` o haber internet para descargarla.

    La primera construcción del motor TensorRT se hace en la propia
    Jetson y puede tardar varios minutos. El archivo `.engine` se guarda
    en `models/` y se reutiliza mientras coincidan el ONNX y TensorRT.

    ## Opción A: desplegar desde Windows PowerShell

    Extraiga el ZIP para obtener el script, pero conserve también el ZIP
    original porque ese es el archivo que se transfiere a la Jetson:

    ```powershell
    $zip = "$env:USERPROFILE\Downloads\jetson_app_yolo26_2.zip"
    $folder = "$env:USERPROFILE\Downloads\jetson_app_yolo26_2"

    Expand-Archive -LiteralPath $zip -DestinationPath $folder -Force
    Set-Location $folder
    Set-ExecutionPolicy -Scope Process Bypass

    .\deploy_to_jetson.ps1 `
        -PackagePath $zip `
        -JetsonIp 192.168.55.1 `
        -JetsonUser artemio
    ```

    El valor por defecto `jetson-app:1.0` permite reutilizar la imagen
    que ya usabas, siempre que contenga OpenCV, CuPy, TensorRT, FastAPI
    y Uvicorn. Si no es compatible, `start_on_jetson.sh` construye
    automáticamente `jetson-yolo26:2.0` con el Dockerfile incluido.

    ## Opción B: instalar directamente en la Jetson

    ```bash
    mkdir -p "$HOME/jetson_app_yolo26_2"
    python3 -m zipfile -e jetson_app_yolo26_2.zip "$HOME/jetson_app_yolo26_2"
    cd "$HOME/jetson_app_yolo26_2"
    bash ./start_on_jetson.sh
    sudo docker logs -f jetson-yolo26
    ```

    Cuando los logs muestren `Uvicorn running`, abra:

    ```text
    http://IP_DE_LA_JETSON:8000
    ```

    Por USB normalmente será:

    ```text
    http://192.168.55.1:8000
    ```

    ## Seleccionar cámara

    Detección automática (prioriza la primera `/dev/video*`):

    ```bash
    bash ./start_on_jetson.sh
    ```

    Cámara USB específica:

    ```bash
    CAMERA_SOURCE=usb CAMERA_DEVICE=/dev/video2 bash ./start_on_jetson.sh
    ```

    Cámara CSI:

    ```bash
    CAMERA_SOURCE=csi CSI_SENSOR_ID=0 bash ./start_on_jetson.sh
    ```

    ## Ajustes útiles

    ```bash
    YOLO_CONF=0.35 CAMERA_FPS=15 JPEG_QUALITY=70 bash ./start_on_jetson.sh
    ```

    El tamaño de entrada (320 por defecto) está fijado al exportar. Para
    cambiarlo a 416 o 640, modifique `MODEL_IMAGE_SIZE` en el notebook y
    vuelva a generar el ZIP.

    ## Diagnóstico

    ```bash
    bash ./diagnose_on_jetson.sh
    curl -s http://127.0.0.1:8000/api/health
    curl -s http://127.0.0.1:8000/api/latest
    ```

    Errores comunes:

    - `CUDA no ve ninguna GPU`: el contenedor no se inició con
      `--runtime nvidia`.
    - `No se pudo abrir la cámara`: pruebe otra `/dev/videoN` o use
      `CAMERA_SOURCE=csi`.
    - `TensorRT no pudo parsear el ONNX`: confirme que la Jetson usa
      JetPack 4.6.1/L4T R32.7.1 y que el ZIP no fue mezclado con modelos
      de una versión anterior.
    - La web abre pero no hay video: revise
      `sudo docker logs -f jetson-yolo26`.

    ## Seguridad

    El puerto 8000 no tiene autenticación. Úselo solo en una red local
    confiable o agregue un proxy/autenticación antes de exponerlo a internet.
    ''',
)

requirements_path = PROJECT_DIR / "requirements-jetson.txt"
wheel_command = [
    sys.executable,
    "-m",
    "pip",
    "download",
    "--dest",
    str(WHEELHOUSE_DIR),
    "--only-binary=:all:",
    "--platform",
    "manylinux2014_aarch64",
    "--implementation",
    "cp",
    "--python-version",
    "3.6.9",
    "--abi",
    "cp36m",
    "-r",
    str(requirements_path),
]

print("Descargando wheels ARM64/Python 3.6...")
subprocess.run(wheel_command, check=True)

wheels = sorted(WHEELHOUSE_DIR.glob("*.whl"))
if len(wheels) < 9:
    raise RuntimeError(
        "Wheelhouse incompleto: solo {} archivos".format(len(wheels))
    )

print("Wheels incluidos:", len(wheels))
for wheel in wheels:
    print(" -", wheel.name)


Descargando wheels ARM64/Python 3.6...
Wheels incluidos: 9
 - aiofiles-0.8.0-py3-none-any.whl
 - click-7.1.2-py2.py3-none-any.whl
 - fastapi-0.63.0-py3-none-any.whl
 - h11-0.13.0-py3-none-any.whl
 - pydantic-1.8.2-py3-none-any.whl
 - starlette-0.13.6-py3-none-any.whl
 - typing_extensions-4.1.1-py3-none-any.whl
 - uvicorn-0.13.4-py3-none-any.whl
 - websockets-9.1-cp36-cp36m-manylinux2014_aarch64.whl


In [10]:
# ============================================================
# AUDITORÍA COMPLETA, MANIFIESTO Y ZIP
# ============================================================

# 1) Debe existir exactamente un ONNX y no debe llevar los nombres
# de los artefactos incompatibles anteriores.
onnx_files = sorted(MODELS_DIR.glob("*.onnx"))
if [path.name for path in onnx_files] != [MODEL_ONNX_NAME]:
    raise RuntimeError(
        "ONNX inesperados en models/: {}".format(
            [path.name for path in onnx_files]
        )
    )

forbidden_names = {
    "yolo26n_raw.onnx",
    "yolo26n_end2end.onnx",
}
package_names = {path.name for path in PROJECT_DIR.rglob("*") if path.is_file()}
if package_names.intersection(forbidden_names):
    raise RuntimeError("Todavía existe un modelo incompatible")

# 2) Todos los Python deben parsear con gramática Python 3.6.
for python_file in sorted(PROJECT_DIR.rglob("*.py")):
    source = python_file.read_text(encoding="utf-8")
    ast.parse(
        source,
        filename=str(python_file),
        feature_version=(3, 6),
    )
    if 'config["labels_file"]f' in source:
        raise RuntimeError("Persistió el carácter f inválido")

# 3) Sintaxis de shell.
for shell_file in sorted(PROJECT_DIR.rglob("*.sh")):
    subprocess.run(
        ["bash", "-n", str(shell_file)],
        check=True,
    )

# 4) Prueba del decodificador con detecciones sintéticas y NMS.
module_path = PROJECT_DIR / "src" / "postprocess.py"
postprocess_namespace = {}
postprocess_source = module_path.read_text(encoding="utf-8")
exec(
    compile(postprocess_source, str(module_path), "exec"),
    postprocess_namespace,
)

synthetic = np.zeros((1, 84, 3), dtype=np.float32)
synthetic[0, 0:4, 0] = [160, 160, 100, 100]
synthetic[0, 4 + 0, 0] = 0.90
synthetic[0, 0:4, 1] = [162, 162, 100, 100]
synthetic[0, 4 + 0, 1] = 0.80
synthetic[0, 0:4, 2] = [70, 70, 40, 40]
synthetic[0, 4 + 2, 2] = 0.75

decoded = postprocess_namespace["decode_yolo26"](
    output=synthetic,
    class_count=80,
    confidence_threshold=0.25,
    nms_threshold=0.45,
    max_detections=50,
    min_box_area=25,
    ratio=1.0,
    pad_x=0,
    pad_y=0,
    frame_width=MODEL_IMAGE_SIZE,
    frame_height=MODEL_IMAGE_SIZE,
)
if len(decoded) != 2:
    raise RuntimeError("NMS sintético falló: {}".format(decoded))

# Nunca empaquetar bytecode creado por otra versión de Python.
for cache_dir in sorted(PROJECT_DIR.rglob("__pycache__"), reverse=True):
    shutil.rmtree(str(cache_dir))
for bytecode in PROJECT_DIR.rglob("*.pyc"):
    bytecode.unlink()
if any(PROJECT_DIR.rglob("*.pyc")):
    raise RuntimeError("Persistió bytecode Python dentro del paquete")

# 5) Comprobar archivos mínimos y configuración.
required_files = [
    "Dockerfile",
    "README.md",
    "start_on_jetson.sh",
    "deploy_to_jetson.ps1",
    "diagnose_on_jetson.sh",
    "src/camera.py",
    "src/model.py",
    "src/postprocess.py",
    "src/server.py",
    "static/index.html",
    "scripts/prepare_engine.py",
    "scripts/entrypoint.sh",
    "models/model_config.json",
    "models/coco.names",
    "models/" + MODEL_ONNX_NAME,
]
missing = [name for name in required_files if not (PROJECT_DIR / name).exists()]
if missing:
    raise RuntimeError("Faltan archivos: {}".format(missing))

config_check = json.loads(
    (MODELS_DIR / "model_config.json").read_text(encoding="utf-8")
)
if config_check["model_file"] != MODEL_ONNX_NAME:
    raise RuntimeError("model_config.json apunta a otro ONNX")
if config_check["onnx_sha256"] != sha256_file(model_path):
    raise RuntimeError("SHA256 de model_config.json incorrecto")

# 6) Manifiesto de integridad.
manifest_lines = []
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file() and path.name != "MANIFEST.sha256":
        relative = path.relative_to(PROJECT_DIR).as_posix()
        manifest_lines.append("{}  {}".format(sha256_file(path), relative))
(PROJECT_DIR / "MANIFEST.sha256").write_text(
    "\n".join(manifest_lines) + "\n",
    encoding="utf-8",
)

# 7) Crear el ZIP sin una carpeta contenedora extra.
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    str(ZIP_PATH),
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(PROJECT_DIR.rglob("*")):
        if path.is_file():
            archive.write(
                str(path),
                path.relative_to(PROJECT_DIR).as_posix(),
            )

# 8) Reabrir y probar el archivo final.
with zipfile.ZipFile(str(ZIP_PATH), "r") as archive:
    bad_member = archive.testzip()
    zip_names = set(archive.namelist())

if bad_member is not None:
    raise RuntimeError("ZIP corrupto en {}".format(bad_member))
if "models/" + MODEL_ONNX_NAME not in zip_names:
    raise RuntimeError("El ZIP no contiene el ONNX")
if any(name.endswith("yolo26n_raw.onnx") for name in zip_names):
    raise RuntimeError("El ZIP contiene yolo26n_raw.onnx")

print("AUDITORÍA_OK")
print("Archivos:", len(zip_names))
print("ZIP:", ZIP_PATH)
print("Tamaño: {:.2f} MiB".format(ZIP_PATH.stat().st_size / 1024 ** 2))


AUDITORÍA_OK
Archivos: 29
ZIP: /content/jetson_app_yolo26_2.zip
Tamaño: 8.78 MiB


In [11]:
# ============================================================
# DESCARGAR EL PAQUETE FINAL
# ============================================================

if RUNNING_IN_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))
else:
    print("Fuera de Colab; archivo disponible en:", ZIP_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Después de descargar

En Windows, extrae el ZIP solo si quieres inspeccionarlo. Para desplegarlo, usa
`deploy_to_jetson.ps1` y pasa el propio ZIP con `-PackagePath`. Durante el primer
arranque sigue los logs con:

```bash
sudo docker logs -f jetson-yolo26
```

No abras ni reemplaces manualmente el ONNX dentro del paquete: el SHA256 se valida y
cualquier motor TensorRT anterior se reconstruye automáticamente si el modelo cambia.
